# DietVA Data Preparation

This notebook prepares the local data files needed by the DietVA agent:

1. **FoodData Central (FDC) Subset** - Local nutrition database
2. **Recipe Database** - SQLite database from recipe-dataset


In [1]:
import json
import sqlite3
from pathlib import Path
import urllib.request

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)


In [2]:
# Download and process the full FDC Foundation Foods dataset
# Replaces the small manual subset with the complete USDA dataset

import zipfile
from collections import Counter

# FDC Nutrient IDs mapping
NUTRIENT_IDS = {
    "calories_kcal": 1008,  # Energy (kcal)
    "protein_g": 1003,       # Protein
    "carbs_g": 1005,        # Carbohydrate, by difference
    "fat_g": 1004,          # Total lipid (fat)
    "fiber_g": 1079,        # Fiber, total dietary
    "sugar_g": 2000,        # Sugars, total including NLEA
}

def generate_tags(description: str, protein_g: float, fiber_g: float, fat_g: float, sugar_g: float) -> list:
    """Generate tags based on description and nutritional content."""
    tags = [word.lower().strip(",.") for word in description.split() if len(word) > 2]
    if protein_g >= 20:
        tags.append("high-protein")
    if fiber_g >= 5:
        tags.append("high-fiber")
    if fat_g <= 3:
        tags.append("low-fat")
    if sugar_g <= 2:
        tags.append("low-sugar")
    return list(set(tags))

def infer_category(description: str, food_category: str = None) -> str:
    """Infer food category from description and FDC category."""
    desc_lower = description.lower()
    
    if food_category and food_category not in ["Foundation Foods", "SR Legacy"]:
        return food_category
    
    if any(word in desc_lower for word in ["chicken", "turkey", "duck", "goose"]):
        return "Poultry"
    elif any(word in desc_lower for word in ["beef", "steak", "ground beef"]):
        return "Beef"
    elif any(word in desc_lower for word in ["pork", "bacon", "ham", "sausage"]):
        return "Pork"
    elif any(word in desc_lower for word in ["fish", "salmon", "tuna", "cod", "tilapia", "shrimp", "crab", "lobster", "oyster", "mussel"]):
        return "Fish & Seafood"
    elif any(word in desc_lower for word in ["egg"]):
        return "Eggs"
    elif any(word in desc_lower for word in ["milk", "yogurt", "cheese", "cream", "butter", "dairy"]):
        return "Dairy"
    elif any(word in desc_lower for word in ["rice", "wheat", "oats", "quinoa", "barley", "pasta", "bread", "cereal", "flour"]):
        return "Grains"
    elif any(word in desc_lower for word in ["apple", "banana", "orange", "berry", "grape", "melon", "peach", "pear", "pineapple", "mango", "kiwi"]):
        return "Fruits"
    elif any(word in desc_lower for word in ["bean", "lentil", "chickpea", "tofu", "tempeh", "edamame", "soy"]):
        return "Legumes & Soy"
    elif any(word in desc_lower for word in ["nut", "almond", "walnut", "cashew", "pecan", "peanut", "seed", "chia", "flax"]):
        return "Nuts & Seeds"
    elif any(word in desc_lower for word in ["oil", "fat", "lard"]):
        return "Oils & Fats"
    elif any(word in desc_lower for word in ["honey", "syrup", "sugar", "sweetener"]):
        return "Sweeteners"
    elif any(word in desc_lower for word in ["chocolate", "candy", "snack", "chip", "cracker"]):
        return "Snacks"
    else:
        return "Vegetables"

def download_fdc_dataset():
    """Download the FDC Foundation Foods dataset."""
    fdc_urls = [
        "https://fdc.nal.usda.gov/fdc-datasets/FoodData_Central_foundation_food_json_2024-04-18.zip",
        "https://fdc.nal.usda.gov/fdc-datasets/FoodData_Central_foundation_food_json.zip",
    ]
    
    zip_path = DATA_DIR / "fdc_foundation_foods.zip"
    json_path = DATA_DIR / "foundationFoods.json"
    
    if json_path.exists():
        print(f"FDC dataset already exists at {json_path}")
        return json_path
    
    print("Downloading FDC Foundation Foods dataset...")
    print("This may take a few minutes (file is ~6-7 MB zipped)...")
    
    for fdc_url in fdc_urls:
        try:
            print(f"Trying: {fdc_url}")
            urllib.request.urlretrieve(fdc_url, zip_path)
            print(f"Downloaded to {zip_path}")
            break
        except Exception as e:
            print(f"  Failed: {e}")
            continue
    else:
        print("ERROR: Could not download from any URL")
        print("\nYou can manually download it from:")
        print("https://fdc.nal.usda.gov/download-datasets.html")
        print("Select 'Foundation Foods' and download the JSON format.")
        return None
    
    try:
        print("Extracting JSON file...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            json_files = [f for f in zip_ref.namelist() if f.endswith('.json')]
            if json_files:
                zip_ref.extract(json_files[0], DATA_DIR)
                extracted_path = DATA_DIR / json_files[0]
                if extracted_path != json_path:
                    extracted_path.rename(json_path)
                print(f"Extracted to {json_path}")
        
        zip_path.unlink()
        print("Download and extraction complete")
        return json_path
    except Exception as e:
        print(f"ERROR extracting: {e}")
        return None

def process_fdc_data(json_path):
    """Process FDC JSON data into our schema."""
    print(f"\nLoading and processing FDC data from {json_path}...")
    
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    foods = data.get("FoundationFoods", data.get("foods", data))
    if not isinstance(foods, list):
        foods = [foods]
    
    print(f"Found {len(foods)} foods in dataset")
    
    processed_foods = []
    skipped = 0
    
    for food in foods:
        try:
            fdc_id = food.get("fdcId")
            description = food.get("description", "").strip()
            
            if not description:
                skipped += 1
                continue
            
            food_category = food.get("foodCategory", {})
            if isinstance(food_category, dict):
                food_category = food_category.get("description", "")
            category = infer_category(description, food_category)
            
            serving_size_g = 100
            if "foodPortions" in food and food["foodPortions"]:
                for portion in food["foodPortions"]:
                    if portion.get("gramWeight", 0) == 100:
                        serving_size_g = 100
                        break
                    elif serving_size_g == 100:
                        serving_size_g = portion.get("gramWeight", 100)
            
            nutrients = {}
            if "foodNutrients" in food:
                for nutrient in food["foodNutrients"]:
                    nutrient_obj = nutrient.get("nutrient", {})
                    nutrient_id = nutrient_obj.get("id") if isinstance(nutrient_obj, dict) else None
                    amount = nutrient.get("amount", 0) or 0
                    
                    for key, nid in NUTRIENT_IDS.items():
                        if nutrient_id == nid:
                            nutrients[key] = float(amount)
                            break
            
            food_entry = {
                "fdc_id": fdc_id,
                "description": description,
                "category": category,
                "serving_size_g": serving_size_g,
                "calories_kcal": nutrients.get("calories_kcal", 0),
                "protein_g": nutrients.get("protein_g", 0),
                "carbs_g": nutrients.get("carbs_g", 0),
                "fat_g": nutrients.get("fat_g", 0),
                "fiber_g": nutrients.get("fiber_g", 0),
                "sugar_g": nutrients.get("sugar_g", 0),
            }
            
            food_entry["tags"] = generate_tags(
                description,
                food_entry["protein_g"],
                food_entry["fiber_g"],
                food_entry["fat_g"],
                food_entry["sugar_g"]
            )
            
            processed_foods.append(food_entry)
            
        except Exception as e:
            skipped += 1
            continue
    
    print(f"Processed {len(processed_foods)} foods (skipped {skipped})")
    return processed_foods

# Download and process
print("=" * 60)
print("FDC Dataset Download and Processing")
print("=" * 60)
json_path = download_fdc_dataset()

if json_path and json_path.exists():
    fdc_foods = process_fdc_data(json_path)
    
    fdc_path = DATA_DIR / "fdc_subset.json"
    with open(fdc_path, "w", encoding="utf-8") as f:
        json.dump(fdc_foods, f, indent=2, ensure_ascii=False)
    
    print(f"\nCreated FDC dataset with {len(fdc_foods)} foods at {fdc_path}")
    
    categories = Counter(f["category"] for f in fdc_foods)
    print(f"\nCategories breakdown (top 15):")
    for cat, count in sorted(categories.items(), key=lambda x: -x[1])[:15]:
        print(f"  {cat}: {count}")
    
    print(f"\nSample entry:\n{json.dumps(fdc_foods[0], indent=2)}")
    
    print("\n" + "=" * 60)
    print("FDC dataset preparation complete")
    print("=" * 60)
else:
    print("\nWARNING: Could not download FDC dataset automatically.")
    print("The notebook will continue with the manual subset below.")
    print("You can manually download from: https://fdc.nal.usda.gov/download-datasets.html")

FDC Dataset Download and Processing
This may take a few minutes (file is ~6-7 MB zipped)...
Trying: https://fdc.nal.usda.gov/fdc-datasets/FoodData_Central_foundation_food_json_2024-04-18.zip
Downloaded to data/fdc_foundation_foods.zip
Extracting JSON file...
Extracted to data/foundationFoods.json
Download and extraction complete

Loading and processing FDC data from data/foundationFoods.json...
Found 287 foods in dataset
Processed 287 foods (skipped 0)

Created FDC dataset with 287 foods at data/fdc_subset.json

Categories breakdown (top 15):
  Vegetables and Vegetable Products: 67
  Dairy and Egg Products: 42
  Legumes and Legume Products: 37
  Fruits and Fruit Juices: 31
  Cereal Grains and Pasta: 27
  Nut and Seed Products: 19
  Beef Products: 12
  Finfish and Shellfish Products: 10
  Fats and Oils: 9
  Poultry Products: 7
  Sausages and Luncheon Meats: 6
  Pork Products: 5
  Restaurant Foods: 4
  Baked Products: 3
  Beverages: 3

Sample entry:
{
  "fdc_id": 321358,
  "description":

## 1. FoodData Central Full Dataset

The USDA FoodData Central provides comprehensive nutrition data. The cell above downloads and processes the **full Foundation Foods dataset** (thousands of foods) instead of using a small manual subset.

**Note:** If the automatic download fails, you can manually download from https://fdc.nal.usda.gov/download-datasets.html and place the JSON file in the `data/` directory. The cell below contains a small fallback subset if needed.


In [ ]:
# Comprehensive FDC subset following the schema from init.md
# In production, download from https://fdc.nal.usda.gov/download-datasets.html
# Schema: fdc_id, description, category, serving_size_g, calories_kcal, protein_g, carbs_g, fat_g, fiber_g, sugar_g, tags

def generate_tags(description: str, protein_g: float, fiber_g: float, fat_g: float, sugar_g: float) -> list:
    """Generate tags based on description and nutritional content."""
    tags = [word.lower().strip(",.") for word in description.split() if len(word) > 2]
    if protein_g >= 20:
        tags.append("high-protein")
    if fiber_g >= 5:
        tags.append("high-fiber")
    if fat_g <= 3:
        tags.append("low-fat")
    if sugar_g <= 2:
        tags.append("low-sugar")
    return list(set(tags))

fdc_foods = [
    # === PROTEINS ===
    {"fdc_id": 171077, "description": "Chicken breast, boneless, skinless, grilled", "category": "Poultry", "serving_size_g": 100, "calories_kcal": 165, "protein_g": 31, "carbs_g": 0, "fat_g": 3.6, "fiber_g": 0, "sugar_g": 0},
    {"fdc_id": 171078, "description": "Chicken thigh, boneless, skinless, roasted", "category": "Poultry", "serving_size_g": 100, "calories_kcal": 209, "protein_g": 26, "carbs_g": 0, "fat_g": 11, "fiber_g": 0, "sugar_g": 0},
    {"fdc_id": 175167, "description": "Turkey breast, roasted", "category": "Poultry", "serving_size_g": 100, "calories_kcal": 135, "protein_g": 30, "carbs_g": 0, "fat_g": 1, "fiber_g": 0, "sugar_g": 0},
    {"fdc_id": 174608, "description": "Beef, ground, 90% lean, pan-browned", "category": "Beef", "serving_size_g": 100, "calories_kcal": 217, "protein_g": 26, "carbs_g": 0, "fat_g": 12, "fiber_g": 0, "sugar_g": 0},
    {"fdc_id": 174609, "description": "Beef, sirloin steak, grilled", "category": "Beef", "serving_size_g": 100, "calories_kcal": 206, "protein_g": 26, "carbs_g": 0, "fat_g": 11, "fiber_g": 0, "sugar_g": 0},
    {"fdc_id": 174610, "description": "Beef, ribeye steak, grilled", "category": "Beef", "serving_size_g": 100, "calories_kcal": 291, "protein_g": 24, "carbs_g": 0, "fat_g": 21, "fiber_g": 0, "sugar_g": 0},
    {"fdc_id": 167512, "description": "Pork tenderloin, roasted", "category": "Pork", "serving_size_g": 100, "calories_kcal": 143, "protein_g": 26, "carbs_g": 0, "fat_g": 3.5, "fiber_g": 0, "sugar_g": 0},
    {"fdc_id": 167513, "description": "Pork chop, bone-in, grilled", "category": "Pork", "serving_size_g": 100, "calories_kcal": 231, "protein_g": 25, "carbs_g": 0, "fat_g": 14, "fiber_g": 0, "sugar_g": 0},
    {"fdc_id": 173686, "description": "Salmon, Atlantic, baked", "category": "Fish & Seafood", "serving_size_g": 100, "calories_kcal": 208, "protein_g": 20, "carbs_g": 0, "fat_g": 13, "fiber_g": 0, "sugar_g": 0},
    {"fdc_id": 173687, "description": "Salmon, sockeye, grilled", "category": "Fish & Seafood", "serving_size_g": 100, "calories_kcal": 216, "protein_g": 27, "carbs_g": 0, "fat_g": 11, "fiber_g": 0, "sugar_g": 0},
    {"fdc_id": 175139, "description": "Tuna, yellowfin, raw", "category": "Fish & Seafood", "serving_size_g": 100, "calories_kcal": 109, "protein_g": 24, "carbs_g": 0, "fat_g": 0.5, "fiber_g": 0, "sugar_g": 0},
    {"fdc_id": 175140, "description": "Tuna, canned in water, drained", "category": "Fish & Seafood", "serving_size_g": 100, "calories_kcal": 116, "protein_g": 26, "carbs_g": 0, "fat_g": 0.8, "fiber_g": 0, "sugar_g": 0},
    {"fdc_id": 174230, "description": "Cod, Atlantic, baked", "category": "Fish & Seafood", "serving_size_g": 100, "calories_kcal": 105, "protein_g": 23, "carbs_g": 0, "fat_g": 0.9, "fiber_g": 0, "sugar_g": 0},
    {"fdc_id": 175168, "description": "Shrimp, steamed", "category": "Fish & Seafood", "serving_size_g": 100, "calories_kcal": 99, "protein_g": 21, "carbs_g": 0.2, "fat_g": 1.1, "fiber_g": 0, "sugar_g": 0},
    {"fdc_id": 175169, "description": "Tilapia, baked", "category": "Fish & Seafood", "serving_size_g": 100, "calories_kcal": 128, "protein_g": 26, "carbs_g": 0, "fat_g": 2.7, "fiber_g": 0, "sugar_g": 0},
    {"fdc_id": 173424, "description": "Tofu, firm, raw", "category": "Legumes & Soy", "serving_size_g": 100, "calories_kcal": 144, "protein_g": 17, "carbs_g": 3, "fat_g": 8, "fiber_g": 2, "sugar_g": 1},
    {"fdc_id": 173425, "description": "Tempeh", "category": "Legumes & Soy", "serving_size_g": 100, "calories_kcal": 192, "protein_g": 20, "carbs_g": 8, "fat_g": 11, "fiber_g": 0, "sugar_g": 0},
    
    # === EGGS & DAIRY ===
    {"fdc_id": 171287, "description": "Egg, whole, boiled", "category": "Eggs", "serving_size_g": 100, "calories_kcal": 155, "protein_g": 13, "carbs_g": 1.1, "fat_g": 11, "fiber_g": 0, "sugar_g": 1.1},
    {"fdc_id": 171288, "description": "Egg, whole, scrambled", "category": "Eggs", "serving_size_g": 100, "calories_kcal": 149, "protein_g": 10, "carbs_g": 2, "fat_g": 11, "fiber_g": 0, "sugar_g": 2},
    {"fdc_id": 171289, "description": "Egg white, cooked", "category": "Eggs", "serving_size_g": 100, "calories_kcal": 52, "protein_g": 11, "carbs_g": 0.7, "fat_g": 0.2, "fiber_g": 0, "sugar_g": 0.7},
    {"fdc_id": 170903, "description": "Greek yogurt, plain, nonfat", "category": "Dairy", "serving_size_g": 100, "calories_kcal": 59, "protein_g": 10, "carbs_g": 3.6, "fat_g": 0.7, "fiber_g": 0, "sugar_g": 3.2},
    {"fdc_id": 170904, "description": "Greek yogurt, plain, whole milk", "category": "Dairy", "serving_size_g": 100, "calories_kcal": 97, "protein_g": 9, "carbs_g": 4, "fat_g": 5, "fiber_g": 0, "sugar_g": 4},
    {"fdc_id": 171265, "description": "Cottage cheese, 1% milkfat", "category": "Dairy", "serving_size_g": 100, "calories_kcal": 72, "protein_g": 12, "carbs_g": 2.7, "fat_g": 1, "fiber_g": 0, "sugar_g": 2.7},
    {"fdc_id": 171266, "description": "Cottage cheese, 2% milkfat", "category": "Dairy", "serving_size_g": 100, "calories_kcal": 86, "protein_g": 11, "carbs_g": 3.4, "fat_g": 2.3, "fiber_g": 0, "sugar_g": 3.4},
    {"fdc_id": 173410, "description": "Milk, whole, 3.25%", "category": "Dairy", "serving_size_g": 100, "calories_kcal": 61, "protein_g": 3.2, "carbs_g": 4.8, "fat_g": 3.3, "fiber_g": 0, "sugar_g": 5},
    {"fdc_id": 173411, "description": "Milk, skim, nonfat", "category": "Dairy", "serving_size_g": 100, "calories_kcal": 34, "protein_g": 3.4, "carbs_g": 5, "fat_g": 0.1, "fiber_g": 0, "sugar_g": 5},
    {"fdc_id": 171270, "description": "Cheese, cheddar", "category": "Dairy", "serving_size_g": 100, "calories_kcal": 403, "protein_g": 25, "carbs_g": 1.3, "fat_g": 33, "fiber_g": 0, "sugar_g": 0.5},
    {"fdc_id": 171271, "description": "Cheese, mozzarella, part-skim", "category": "Dairy", "serving_size_g": 100, "calories_kcal": 280, "protein_g": 28, "carbs_g": 3.1, "fat_g": 17, "fiber_g": 0, "sugar_g": 1.1},
    {"fdc_id": 171272, "description": "Cheese, parmesan, grated", "category": "Dairy", "serving_size_g": 100, "calories_kcal": 431, "protein_g": 38, "carbs_g": 4.1, "fat_g": 29, "fiber_g": 0, "sugar_g": 0.9},
    {"fdc_id": 171273, "description": "Cheese, feta", "category": "Dairy", "serving_size_g": 100, "calories_kcal": 264, "protein_g": 14, "carbs_g": 4, "fat_g": 21, "fiber_g": 0, "sugar_g": 4},
    
    # === GRAINS & CARBS ===
    {"fdc_id": 168880, "description": "Rice, brown, cooked", "category": "Grains", "serving_size_g": 100, "calories_kcal": 123, "protein_g": 2.7, "carbs_g": 25.6, "fat_g": 1, "fiber_g": 1.8, "sugar_g": 0.4},
    {"fdc_id": 168881, "description": "Rice, white, cooked", "category": "Grains", "serving_size_g": 100, "calories_kcal": 130, "protein_g": 2.7, "carbs_g": 28, "fat_g": 0.3, "fiber_g": 0.4, "sugar_g": 0},
    {"fdc_id": 169705, "description": "Quinoa, cooked", "category": "Grains", "serving_size_g": 100, "calories_kcal": 120, "protein_g": 4.4, "carbs_g": 21, "fat_g": 1.9, "fiber_g": 2.8, "sugar_g": 0.9},
    {"fdc_id": 168893, "description": "Oatmeal, regular, cooked", "category": "Grains", "serving_size_g": 100, "calories_kcal": 71, "protein_g": 2.5, "carbs_g": 12, "fat_g": 1.5, "fiber_g": 1.7, "sugar_g": 0.5},
    {"fdc_id": 168894, "description": "Oats, rolled, dry", "category": "Grains", "serving_size_g": 100, "calories_kcal": 379, "protein_g": 13, "carbs_g": 67, "fat_g": 6.5, "fiber_g": 10, "sugar_g": 1},
    {"fdc_id": 168917, "description": "Pasta, whole wheat, cooked", "category": "Grains", "serving_size_g": 100, "calories_kcal": 124, "protein_g": 5.3, "carbs_g": 25, "fat_g": 0.5, "fiber_g": 4.5, "sugar_g": 0.6},
    {"fdc_id": 168918, "description": "Pasta, white, cooked", "category": "Grains", "serving_size_g": 100, "calories_kcal": 131, "protein_g": 5, "carbs_g": 25, "fat_g": 1.1, "fiber_g": 1.8, "sugar_g": 0.6},
    {"fdc_id": 172686, "description": "Bread, whole wheat", "category": "Grains", "serving_size_g": 100, "calories_kcal": 247, "protein_g": 13, "carbs_g": 41, "fat_g": 3.4, "fiber_g": 7, "sugar_g": 6},
    {"fdc_id": 172687, "description": "Bread, white", "category": "Grains", "serving_size_g": 100, "calories_kcal": 265, "protein_g": 9, "carbs_g": 49, "fat_g": 3.2, "fiber_g": 2.7, "sugar_g": 5},
    {"fdc_id": 170189, "description": "Potato, baked, flesh and skin", "category": "Vegetables", "serving_size_g": 100, "calories_kcal": 93, "protein_g": 2.5, "carbs_g": 21, "fat_g": 0.1, "fiber_g": 2.2, "sugar_g": 1.7},
    {"fdc_id": 170190, "description": "Sweet potato, baked", "category": "Vegetables", "serving_size_g": 100, "calories_kcal": 90, "protein_g": 2, "carbs_g": 21, "fat_g": 0.1, "fiber_g": 3.3, "sugar_g": 6.5},
    
    # === VEGETABLES ===
    {"fdc_id": 169967, "description": "Broccoli, steamed", "category": "Vegetables", "serving_size_g": 100, "calories_kcal": 35, "protein_g": 2.4, "carbs_g": 7.2, "fat_g": 0.4, "fiber_g": 3.3, "sugar_g": 1.4},
    {"fdc_id": 169968, "description": "Broccoli, raw", "category": "Vegetables", "serving_size_g": 100, "calories_kcal": 34, "protein_g": 2.8, "carbs_g": 7, "fat_g": 0.4, "fiber_g": 2.6, "sugar_g": 1.7},
    {"fdc_id": 170393, "description": "Spinach, raw", "category": "Vegetables", "serving_size_g": 100, "calories_kcal": 23, "protein_g": 2.9, "carbs_g": 3.6, "fat_g": 0.4, "fiber_g": 2.2, "sugar_g": 0.4},
    {"fdc_id": 170394, "description": "Spinach, cooked, boiled", "category": "Vegetables", "serving_size_g": 100, "calories_kcal": 23, "protein_g": 3, "carbs_g": 3.8, "fat_g": 0.3, "fiber_g": 2.4, "sugar_g": 0.4},
    {"fdc_id": 170417, "description": "Kale, raw", "category": "Vegetables", "serving_size_g": 100, "calories_kcal": 49, "protein_g": 4.3, "carbs_g": 9, "fat_g": 0.9, "fiber_g": 3.6, "sugar_g": 2.3},
    {"fdc_id": 169228, "description": "Carrot, raw", "category": "Vegetables", "serving_size_g": 100, "calories_kcal": 41, "protein_g": 0.9, "carbs_g": 10, "fat_g": 0.2, "fiber_g": 2.8, "sugar_g": 4.7},
    {"fdc_id": 169229, "description": "Carrot, cooked, boiled", "category": "Vegetables", "serving_size_g": 100, "calories_kcal": 35, "protein_g": 0.8, "carbs_g": 8, "fat_g": 0.2, "fiber_g": 3, "sugar_g": 3.5},
    {"fdc_id": 169986, "description": "Cauliflower, raw", "category": "Vegetables", "serving_size_g": 100, "calories_kcal": 25, "protein_g": 1.9, "carbs_g": 5, "fat_g": 0.3, "fiber_g": 2, "sugar_g": 1.9},
    {"fdc_id": 170079, "description": "Bell pepper, red, raw", "category": "Vegetables", "serving_size_g": 100, "calories_kcal": 31, "protein_g": 1, "carbs_g": 6, "fat_g": 0.3, "fiber_g": 2.1, "sugar_g": 4.2},
    {"fdc_id": 170080, "description": "Bell pepper, green, raw", "category": "Vegetables", "serving_size_g": 100, "calories_kcal": 20, "protein_g": 0.9, "carbs_g": 4.6, "fat_g": 0.2, "fiber_g": 1.7, "sugar_g": 2.4},
    {"fdc_id": 170457, "description": "Tomato, raw", "category": "Vegetables", "serving_size_g": 100, "calories_kcal": 18, "protein_g": 0.9, "carbs_g": 3.9, "fat_g": 0.2, "fiber_g": 1.2, "sugar_g": 2.6},
    {"fdc_id": 169213, "description": "Cucumber, raw, with peel", "category": "Vegetables", "serving_size_g": 100, "calories_kcal": 15, "protein_g": 0.7, "carbs_g": 3.6, "fat_g": 0.1, "fiber_g": 0.5, "sugar_g": 1.7},
    {"fdc_id": 170470, "description": "Zucchini, raw", "category": "Vegetables", "serving_size_g": 100, "calories_kcal": 17, "protein_g": 1.2, "carbs_g": 3.1, "fat_g": 0.3, "fiber_g": 1, "sugar_g": 2.5},
    {"fdc_id": 170151, "description": "Onion, raw", "category": "Vegetables", "serving_size_g": 100, "calories_kcal": 40, "protein_g": 1.1, "carbs_g": 9.3, "fat_g": 0.1, "fiber_g": 1.7, "sugar_g": 4.2},
    {"fdc_id": 169230, "description": "Celery, raw", "category": "Vegetables", "serving_size_g": 100, "calories_kcal": 14, "protein_g": 0.7, "carbs_g": 3, "fat_g": 0.2, "fiber_g": 1.6, "sugar_g": 1.3},
    {"fdc_id": 168409, "description": "Asparagus, cooked, boiled", "category": "Vegetables", "serving_size_g": 100, "calories_kcal": 22, "protein_g": 2.4, "carbs_g": 4.1, "fat_g": 0.2, "fiber_g": 2, "sugar_g": 1.3},
    {"fdc_id": 170172, "description": "Green beans, cooked", "category": "Vegetables", "serving_size_g": 100, "calories_kcal": 35, "protein_g": 1.9, "carbs_g": 8, "fat_g": 0.3, "fiber_g": 3.2, "sugar_g": 1.5},
    {"fdc_id": 170391, "description": "Mushrooms, white, raw", "category": "Vegetables", "serving_size_g": 100, "calories_kcal": 22, "protein_g": 3.1, "carbs_g": 3.3, "fat_g": 0.3, "fiber_g": 1, "sugar_g": 2},
    {"fdc_id": 169246, "description": "Lettuce, romaine, raw", "category": "Vegetables", "serving_size_g": 100, "calories_kcal": 17, "protein_g": 1.2, "carbs_g": 3.3, "fat_g": 0.3, "fiber_g": 2.1, "sugar_g": 1.2},
    {"fdc_id": 168483, "description": "Brussels sprouts, cooked", "category": "Vegetables", "serving_size_g": 100, "calories_kcal": 36, "protein_g": 2.6, "carbs_g": 7, "fat_g": 0.5, "fiber_g": 2.6, "sugar_g": 1.9},
    {"fdc_id": 168420, "description": "Avocado, raw", "category": "Vegetables", "serving_size_g": 100, "calories_kcal": 160, "protein_g": 2, "carbs_g": 9, "fat_g": 15, "fiber_g": 7, "sugar_g": 0.7},
    
    # === FRUITS ===
    {"fdc_id": 173944, "description": "Banana, raw", "category": "Fruits", "serving_size_g": 100, "calories_kcal": 89, "protein_g": 1.1, "carbs_g": 23, "fat_g": 0.3, "fiber_g": 2.6, "sugar_g": 12},
    {"fdc_id": 168203, "description": "Apple, raw, with skin", "category": "Fruits", "serving_size_g": 100, "calories_kcal": 52, "protein_g": 0.3, "carbs_g": 14, "fat_g": 0.2, "fiber_g": 2.4, "sugar_g": 10},
    {"fdc_id": 167762, "description": "Orange, raw", "category": "Fruits", "serving_size_g": 100, "calories_kcal": 47, "protein_g": 0.9, "carbs_g": 12, "fat_g": 0.1, "fiber_g": 2.4, "sugar_g": 9.4},
    {"fdc_id": 167764, "description": "Strawberries, raw", "category": "Fruits", "serving_size_g": 100, "calories_kcal": 32, "protein_g": 0.7, "carbs_g": 7.7, "fat_g": 0.3, "fiber_g": 2, "sugar_g": 4.9},
    {"fdc_id": 171711, "description": "Blueberries, raw", "category": "Fruits", "serving_size_g": 100, "calories_kcal": 57, "protein_g": 0.7, "carbs_g": 14, "fat_g": 0.3, "fiber_g": 2.4, "sugar_g": 10},
    {"fdc_id": 167753, "description": "Grapes, red, raw", "category": "Fruits", "serving_size_g": 100, "calories_kcal": 69, "protein_g": 0.7, "carbs_g": 18, "fat_g": 0.2, "fiber_g": 0.9, "sugar_g": 16},
    {"fdc_id": 169910, "description": "Watermelon, raw", "category": "Fruits", "serving_size_g": 100, "calories_kcal": 30, "protein_g": 0.6, "carbs_g": 7.6, "fat_g": 0.2, "fiber_g": 0.4, "sugar_g": 6.2},
    {"fdc_id": 167755, "description": "Pineapple, raw", "category": "Fruits", "serving_size_g": 100, "calories_kcal": 50, "protein_g": 0.5, "carbs_g": 13, "fat_g": 0.1, "fiber_g": 1.4, "sugar_g": 10},
    {"fdc_id": 169926, "description": "Mango, raw", "category": "Fruits", "serving_size_g": 100, "calories_kcal": 60, "protein_g": 0.8, "carbs_g": 15, "fat_g": 0.4, "fiber_g": 1.6, "sugar_g": 14},
    {"fdc_id": 169943, "description": "Peach, raw", "category": "Fruits", "serving_size_g": 100, "calories_kcal": 39, "protein_g": 0.9, "carbs_g": 10, "fat_g": 0.3, "fiber_g": 1.5, "sugar_g": 8.4},
    {"fdc_id": 168162, "description": "Raspberries, raw", "category": "Fruits", "serving_size_g": 100, "calories_kcal": 52, "protein_g": 1.2, "carbs_g": 12, "fat_g": 0.7, "fiber_g": 6.5, "sugar_g": 4.4},
    {"fdc_id": 173946, "description": "Kiwi, raw", "category": "Fruits", "serving_size_g": 100, "calories_kcal": 61, "protein_g": 1.1, "carbs_g": 15, "fat_g": 0.5, "fiber_g": 3, "sugar_g": 9},
    {"fdc_id": 173945, "description": "Pear, raw", "category": "Fruits", "serving_size_g": 100, "calories_kcal": 57, "protein_g": 0.4, "carbs_g": 15, "fat_g": 0.1, "fiber_g": 3.1, "sugar_g": 10},
    
    # === LEGUMES ===
    {"fdc_id": 175197, "description": "Chickpeas, canned, drained", "category": "Legumes & Soy", "serving_size_g": 100, "calories_kcal": 139, "protein_g": 7.5, "carbs_g": 23, "fat_g": 2.6, "fiber_g": 6, "sugar_g": 4},
    {"fdc_id": 173757, "description": "Black beans, canned, drained", "category": "Legumes & Soy", "serving_size_g": 100, "calories_kcal": 91, "protein_g": 6, "carbs_g": 16, "fat_g": 0.4, "fiber_g": 7, "sugar_g": 0.3},
    {"fdc_id": 175198, "description": "Lentils, cooked", "category": "Legumes & Soy", "serving_size_g": 100, "calories_kcal": 116, "protein_g": 9, "carbs_g": 20, "fat_g": 0.4, "fiber_g": 8, "sugar_g": 1.8},
    {"fdc_id": 173756, "description": "Kidney beans, canned, drained", "category": "Legumes & Soy", "serving_size_g": 100, "calories_kcal": 105, "protein_g": 7, "carbs_g": 18, "fat_g": 0.5, "fiber_g": 6.5, "sugar_g": 2},
    {"fdc_id": 174271, "description": "Edamame, shelled, cooked", "category": "Legumes & Soy", "serving_size_g": 100, "calories_kcal": 121, "protein_g": 12, "carbs_g": 9, "fat_g": 5, "fiber_g": 5, "sugar_g": 2},
    {"fdc_id": 172431, "description": "Peanut butter, smooth", "category": "Nuts & Seeds", "serving_size_g": 100, "calories_kcal": 588, "protein_g": 25, "carbs_g": 20, "fat_g": 50, "fiber_g": 6, "sugar_g": 9},
    
    # === NUTS & SEEDS ===
    {"fdc_id": 170567, "description": "Almonds, raw", "category": "Nuts & Seeds", "serving_size_g": 100, "calories_kcal": 579, "protein_g": 21, "carbs_g": 22, "fat_g": 50, "fiber_g": 12, "sugar_g": 4.4},
    {"fdc_id": 170182, "description": "Walnuts, raw", "category": "Nuts & Seeds", "serving_size_g": 100, "calories_kcal": 654, "protein_g": 15, "carbs_g": 14, "fat_g": 65, "fiber_g": 6.7, "sugar_g": 2.6},
    {"fdc_id": 170571, "description": "Cashews, raw", "category": "Nuts & Seeds", "serving_size_g": 100, "calories_kcal": 553, "protein_g": 18, "carbs_g": 30, "fat_g": 44, "fiber_g": 3.3, "sugar_g": 6},
    {"fdc_id": 170178, "description": "Pecans, raw", "category": "Nuts & Seeds", "serving_size_g": 100, "calories_kcal": 691, "protein_g": 9, "carbs_g": 14, "fat_g": 72, "fiber_g": 9.6, "sugar_g": 4},
    {"fdc_id": 170554, "description": "Chia seeds, dried", "category": "Nuts & Seeds", "serving_size_g": 100, "calories_kcal": 486, "protein_g": 17, "carbs_g": 42, "fat_g": 31, "fiber_g": 34, "sugar_g": 0},
    {"fdc_id": 170148, "description": "Flaxseeds, ground", "category": "Nuts & Seeds", "serving_size_g": 100, "calories_kcal": 534, "protein_g": 18, "carbs_g": 29, "fat_g": 42, "fiber_g": 27, "sugar_g": 1.6},
    {"fdc_id": 170187, "description": "Sunflower seeds, kernels, dry roasted", "category": "Nuts & Seeds", "serving_size_g": 100, "calories_kcal": 582, "protein_g": 19, "carbs_g": 24, "fat_g": 51, "fiber_g": 11, "sugar_g": 2.7},
    {"fdc_id": 170184, "description": "Pumpkin seeds, kernels, roasted", "category": "Nuts & Seeds", "serving_size_g": 100, "calories_kcal": 446, "protein_g": 19, "carbs_g": 54, "fat_g": 19, "fiber_g": 18, "sugar_g": 1.3},
    
    # === OILS & FATS ===
    {"fdc_id": 171413, "description": "Olive oil, extra virgin", "category": "Oils & Fats", "serving_size_g": 100, "calories_kcal": 884, "protein_g": 0, "carbs_g": 0, "fat_g": 100, "fiber_g": 0, "sugar_g": 0},
    {"fdc_id": 171411, "description": "Coconut oil", "category": "Oils & Fats", "serving_size_g": 100, "calories_kcal": 862, "protein_g": 0, "carbs_g": 0, "fat_g": 100, "fiber_g": 0, "sugar_g": 0},
    {"fdc_id": 173578, "description": "Butter, salted", "category": "Oils & Fats", "serving_size_g": 100, "calories_kcal": 717, "protein_g": 0.9, "carbs_g": 0.1, "fat_g": 81, "fiber_g": 0, "sugar_g": 0.1},
    
    # === BEVERAGES & MISC ===
    {"fdc_id": 174832, "description": "Honey", "category": "Sweeteners", "serving_size_g": 100, "calories_kcal": 304, "protein_g": 0.3, "carbs_g": 82, "fat_g": 0, "fiber_g": 0.2, "sugar_g": 82},
    {"fdc_id": 170000, "description": "Maple syrup, pure", "category": "Sweeteners", "serving_size_g": 100, "calories_kcal": 260, "protein_g": 0, "carbs_g": 67, "fat_g": 0.1, "fiber_g": 0, "sugar_g": 60},
    {"fdc_id": 171891, "description": "Dark chocolate, 70-85% cacao", "category": "Snacks", "serving_size_g": 100, "calories_kcal": 598, "protein_g": 8, "carbs_g": 46, "fat_g": 43, "fiber_g": 11, "sugar_g": 24},
]

# Add tags to each food item
for food in fdc_foods:
    food["tags"] = generate_tags(
        food["description"],
        food["protein_g"],
        food["fiber_g"],
        food["fat_g"],
        food["sugar_g"]
    )

fdc_path = DATA_DIR / "fdc_subset.json"
with open(fdc_path, "w", encoding="utf-8") as f:
    json.dump(fdc_foods, f, indent=2, ensure_ascii=False)

print(f"Created FDC subset with {len(fdc_foods)} foods at {fdc_path}")
print(f"\nCategories: {set(f['category'] for f in fdc_foods)}")
print(f"\nSample entry:\n{json.dumps(fdc_foods[0], indent=2)}")


## 2. Recipe Database

The recipe database has been downloaded from the `recipe-dataset` repository (13k-recipes.db).
Let's verify it and optionally create indexes for faster searching.


In [ ]:
# Verify and index the downloaded recipe database
# Download command (if not already done):
# curl -L https://github.com/josephrmartinez/recipe-dataset/raw/main/13k-recipes.db -o data/recipes.db

db_path = DATA_DIR / "recipes.db"

if not db_path.exists():
    print(f"ERROR: Recipe database not found at {db_path}")
    print("Download it with: curl -L https://github.com/josephrmartinez/recipe-dataset/raw/main/13k-recipes.db -o data/recipes.db")
else:
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # Check schema
    cursor.execute("SELECT sql FROM sqlite_master WHERE type='table' AND name='recipes'")
    schema = cursor.fetchone()
    print(f"Recipe table schema:\n{schema[0]}\n")
    
    # Count recipes
    cursor.execute("SELECT COUNT(*) FROM recipes")
    count = cursor.fetchone()[0]
    print(f"Total recipes: {count}")
    
    # Create indexes for faster searching (if they don't exist)
    print("\nCreating indexes for faster searching...")
    try:
        cursor.execute("CREATE INDEX IF NOT EXISTS idx_recipes_title ON recipes(Title)")
        cursor.execute("CREATE INDEX IF NOT EXISTS idx_recipes_ingredients ON recipes(Ingredients)")
        conn.commit()
        print("Indexes created successfully!")
    except sqlite3.Error as e:
        print(f"Index creation note: {e}")
    
    # Show sample recipes
    print("\nSample recipes:")
    cursor.execute("SELECT id, Title, substr(Ingredients, 1, 80) as ing_preview FROM recipes LIMIT 5")
    for row in cursor.fetchall():
        print(f"  [{row[0]}] {row[1]}")
        print(f"      Ingredients: {row[2]}...")
    
    conn.close()


## 3. Verify Data


In [ ]:
# Verify FDC data
with open(DATA_DIR / "fdc_subset.json", encoding="utf-8") as f:
    foods = json.load(f)

print(f"FDC subset contains {len(foods)} foods")
print(f"\nCategories breakdown:")
from collections import Counter
categories = Counter(f["category"] for f in foods)
for cat, count in sorted(categories.items(), key=lambda x: -x[1]):
    print(f"  {cat}: {count}")

print(f"\nSample high-protein foods:")
high_protein = [f for f in foods if "high-protein" in f.get("tags", [])][:5]
for f in high_protein:
    print(f"  - {f['description']}: {f['protein_g']}g protein")


In [ ]:
# Test recipe search functionality
conn = sqlite3.connect(DATA_DIR / "recipes.db")
cursor = conn.cursor()

# Test search for "chicken"
print("Search test for 'chicken':")
cursor.execute("""
    SELECT Title, substr(Ingredients, 1, 100) 
    FROM recipes 
    WHERE Title LIKE '%chicken%' OR Ingredients LIKE '%chicken%'
    LIMIT 3
""")
for title, ingredients in cursor.fetchall():
    print(f"  - {title}")
    print(f"    {ingredients}...")

# Test search for vegetarian options (no meat keywords)
print("\nSearch test for vegetarian-friendly (tofu):")
cursor.execute("""
    SELECT Title, substr(Ingredients, 1, 100) 
    FROM recipes 
    WHERE Ingredients LIKE '%tofu%'
    LIMIT 3
""")
for title, ingredients in cursor.fetchall():
    print(f"  - {title}")
    
conn.close()
print("\nData preparation complete!")


## 4. (Optional) Download Full Datasets

Uncomment the cells below to download full datasets.


In [ ]:
# TODO: Add code to download full FDC dataset
# URL: https://fdc.nal.usda.gov/download-datasets.html

# TODO: Add code to download and import recipe-dataset
# URL: https://github.com/Glorf/recipenlg
